# ⭐ Review Analyst: Structured Review Analysis with Routed Replies

## Learning Objectives
In this notebook, you will learn:
1. **Structured review analysis** - Turn each free-text review into a typed record (summary, positives, negatives, sentiment, emotions) and a table
2. **Checking the analysis** - Compare the model's sentiment with hand labels before trusting it downstream
3. **Replies routed by sentiment** - Pick the thank-you or the apology email writer in code from the structured sentiment, and check each email's required parts
4. **An instruction router for analysts** - Send a free-text request ("summarize this", "draft a reply") to the right chain, with a fallback for anything else

## Where this fits
Structured output (`04_langchain_agents_and_middleware/03_structured_output.ipynb`), prompt chaining (Module 05) and routing (`06_routing/03_routing_lcel_runnable_branch.ipynb`) are the building blocks. This notebook combines them into one customer-review tool of the kind a support team actually uses.

## Prerequisites
- Structured output, chaining and routing from Modules 04–06 (reading only)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

Load the API key from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: review analysis and reply writing
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🗂️ Part 1: The Task and the Reviews


You are building an AI system to be able to look at customer reviews and do some complex analysis. For each review, get the model to do the following:

  - Summarize the review. The summary should be at most 3 lines.
  - Highlight both the positives and negatives
  - Display the overall sentiment of the review (positive, negative, neutral)
  - Display a list of 3 - 5 emotions expressed by the customer in the review
  - If the sentiment is positive or neutral write an email and thank them for the review
  - If the sentiment is negative apologize and write an email with an appropriate response

Get the response in a structured format that code can use

Six product reviews: five from an electronics store and a longer one about an office lamp that mixes problems with praise. Each review has a **hand label** for its overall sentiment, so the model's analysis can be checked.

In [ ]:
# ============================================================================
# THE REVIEWS + HAND LABELS
# ============================================================================
reviews = [
    """
    Just received the Bluetooth speaker I ordered for beach outings, and it's fantastic.
    The sound quality is impressively clear with just the right amount of bass.
    It's also waterproof, which tested true during a recent splashing incident.
    Though it's compact, the volume can really fill the space.
    The price was a bargain for such high-quality sound.
    Shipping was also on point, arriving two days early in secure packaging.
    """,
    """
    Purchased a new gaming keyboard because of its rave reviews about responsiveness and backlighting.
    It hasn't disappointed. The keys have a satisfying click and the LED colors are vibrant,
    enhancing my gaming experience significantly. Price-wise, it's quite competitive,
    and I feel like I got a good deal. The delivery was swift, and it came well-protected,
    ensuring no damage during transport.
    """,
    """
    Ordered a set of wireless earbuds for running, and they've been a letdown.
    The sound constantly cuts out, and the fit is uncomfortable after only a few minutes of use.
    They advertised a 12-hour battery life, but I'm barely getting four hours.
    Considering the cost, I expected better quality and performance.
    They did arrive on time, but the positives end there. I'm already looking into a return.
    """,
    """
    The tablet stand I bought was touted as being sturdy and adjustable,
    but it's anything but. It wobbles with the slightest touch,
    and the angles are not holding up as promised. It feels like a breeze could knock it over.
    It was also pricier than others I've seen, which adds to the disappointment.
    It did arrive promptly, but what's the use if the product doesn't meet basic expectations?
    """,
    """
    Needed a new kitchen blender, but this model has been a nightmare.
    It's supposed to handle various foods, but it struggles with anything tougher than cooked vegetables.
    It's also incredibly noisy, and the 'easy-clean' feature is a joke; food gets stuck under the blades constantly.
    I thought the brand meant quality, but this product has proven me wrong.
    Plus, it arrived three days late. Definitely not worth the expense.
    """
]

reviews.append("""
    Required a stylish lamp for my office space, and this particular one
    came with added storage at a reasonable price.
    The delivery was surprisingly quick, arriving within just two days.
    However, the pull string for the lamp suffered damage during transit.
    To my relief, the company promptly dispatched a replacement,
    which arrived within a few days. Assembly was a breeze.
    Then, I encountered an issue with a missing component,
    but their support team responded swiftly and provided the missing part.
    It appears to be a commendable company that genuinely values its
    customers and the quality of its products.
""")

# Overall sentiment, labelled by a person (the lamp review is positive despite its problems)
LABELS = ["positive", "positive", "negative", "negative", "negative", "positive"]

print(f"✅ {len(reviews)} reviews, labels: {LABELS}")

---
## 🧾 Part 2: Structured Review Analysis

`with_structured_output(ReviewAnalysis)` makes the model return an instance of the schema directly — no format instructions pasted into the prompt, no output parser (Module 04 explains the mechanics). `sentiment` is a `Literal`, so it can only be one of three values that code can branch on. `.batch()` analyses all reviews concurrently.

In [ ]:
# ============================================================================
# REVIEW SCHEMA + ANALYSIS CHAIN
# ============================================================================
from typing import Literal

import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class ReviewAnalysis(BaseModel):
    summary: str = Field(description="Summary of the review in at most 3 lines")
    positives: list[str] = Field(description="Positive aspects mentioned, if any (max 3)")
    negatives: list[str] = Field(description="Negative aspects mentioned, if any (max 3)")
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="Overall sentiment of the review")
    emotions: list[str] = Field(description="3 to 5 emotions expressed by the customer")


analysis_prompt = ChatPromptTemplate.from_template(
    "Analyze the customer review below.\n\nReview:\n{review}"
)
analyzer = analysis_prompt | llm.with_structured_output(ReviewAnalysis)

analyses = analyzer.batch([{"review": r} for r in reviews])

df = pd.DataFrame([a.model_dump() for a in analyses])
df["label"] = LABELS
print(df[["summary", "sentiment", "label", "emotions"]].to_string(max_colwidth=60))

In [ ]:
# ============================================================================
# CHECK THE SENTIMENT AGAINST THE HAND LABELS
# ============================================================================
agree = (df["sentiment"] == df["label"]).sum()
print(f"Sentiment agrees with the hand label on {agree}/{len(df)} reviews")
for i, row in df[df["sentiment"] != df["label"]].iterrows():
    print(f"  review {i}: model says {row['sentiment']!r}, label is {row['label']!r} — {row['summary'][:80]}")

Any disagreement here would flow straight into the next step, where sentiment decides which email a customer receives — so check it first. Mixed reviews like the lamp one are where models and people most often differ.

---
## ✉️ Part 3: Replies Routed by Sentiment

Each review gets an email. Because `sentiment` is a structured field, routing is a dictionary lookup in code — no LLM call to decide the route: positive and neutral reviews get a thank-you, negative ones an apology. Both writers share the same rules (addressed to "Dear Customer", signed "Service Agent"), and code checks every email for them.

In [ ]:
# ============================================================================
# TWO EMAIL WRITERS + ROUTING IN CODE
# ============================================================================
from langchain_core.output_parsers import StrOutputParser

EMAIL_RULES = """Address the email to "Dear Customer" and sign it "Service Agent".
Mention at least one specific point from the review. Keep it under 120 words."""

thank_you_chain = ChatPromptTemplate.from_template(
    "Write a short email thanking the customer for this review.\n" + EMAIL_RULES +
    "\n\nReview:\n{review}") | llm | StrOutputParser()
apology_chain = ChatPromptTemplate.from_template(
    "Write a short email apologizing to the customer for the problems in this review and "
    "offering help (replacement, refund or support).\n" + EMAIL_RULES +
    "\n\nReview:\n{review}") | llm | StrOutputParser()

REPLY_BY_SENTIMENT = {"positive": thank_you_chain, "neutral": thank_you_chain,
                      "negative": apology_chain}

emails = [REPLY_BY_SENTIMENT[a.sentiment].invoke({"review": r}) for a, r in zip(analyses, reviews)]
print(f"✅ {len(emails)} emails written — routes used: {sorted({a.sentiment for a in analyses})}")
print("\n--- first email ---\n" + emails[0])
print("\n--- first apology ---\n" + emails[[a.sentiment for a in analyses].index("negative")])

In [ ]:
# ============================================================================
# CHECK EVERY EMAIL IN CODE
# ============================================================================
def email_problems(sentiment: str, email: str) -> list[str]:
    """Names of the rules an email breaks (empty list = all good)."""
    text = email.lower()
    apologizes = "apolog" in text or "sorry" in text
    checks = {
        "greeting": "dear customer" in text,
        "signature": "service agent" in text,
        "tone": apologizes if sentiment == "negative" else ("thank" in text and not apologizes),
        "length": len(email.split()) <= 140,
    }
    return [name for name, ok in checks.items() if not ok]


for i, (a, email) in enumerate(zip(analyses, emails)):
    failed = email_problems(a.sentiment, email)
    print(f"{'✅' if not failed else '❌'} review {i} ({a.sentiment}): {len(email.split())} words"
          + (f" — failed: {failed}" if failed else ""))

# The same check on an email sent down the wrong route (apology for a happy customer, no signature)
wrong = "Dear Customer, we are sorry to hear about your experience. Regards, Support"
print(f"\nWrong-route example → failed: {email_problems('positive', wrong)}")

---
## 🧭 Part 4: An Instruction Router for Analysts

The support team also asks ad-hoc questions about a single review: *"summarize this"*, *"what's the sentiment?"*, *"draft a reply"*. A router classifies the instruction into one of four intents with structured output, and a dictionary sends it to the right chain. The fourth intent, `other`, catches requests the tool does not handle — without it, an unrelated request would be forced into one of the three chains.

In [ ]:
# ============================================================================
# INTENT ROUTER + DISPATCH TABLE
# ============================================================================
class Intent(BaseModel):
    intent: Literal["summarize", "sentiment", "email", "other"] = Field(
        description="summarize: summary of the review; sentiment: its sentiment; "
                    "email: a reply to the customer; other: anything else")


router = ChatPromptTemplate.from_template(
    "Classify this instruction about a customer review.\n\nInstruction: {instruction}"
) | llm.with_structured_output(Intent)


def summarize(review: str) -> str:
    return analyzer.invoke({"review": review}).summary


def sentiment(review: str) -> str:
    return analyzer.invoke({"review": review}).sentiment


def email(review: str) -> str:
    return REPLY_BY_SENTIMENT[sentiment(review)].invoke({"review": review})


HANDLERS = {"summarize": summarize, "sentiment": sentiment, "email": email,
            "other": lambda review: "Sorry, I can only summarize a review, give its sentiment or draft a reply."}


def handle(instruction: str, review: str) -> tuple[str, str]:
    intent = router.invoke({"instruction": instruction}).intent
    return intent, HANDLERS[intent](https://github.com/Sourav692/ai-engineering-demystified-06-agent-fundamentals/tree/main/modules/14_applied_single_agent_builds/notebooks/review)


print(f"✅ Router intents: {list(HANDLERS)}")

In [ ]:
# ============================================================================
# TEST THE ROUTER: one instruction per intent, including an out-of-scope one
# ============================================================================
lamp_review = reviews[-1]
tests = [
    ("Give me the gist of this review in two lines", "summarize"),
    ("Is this customer happy or not?", "sentiment"),
    ("Write back to this customer", "email"),
    ("What's the capital of France?", "other"),
]
for instruction, expected in tests:
    intent, output = handle(instruction, lamp_review)
    print(f"{'✅' if intent == expected else '❌'} {instruction!r} → {intent}\n   {output[:200]}\n")

---
## 📝 Summary

In this notebook, we learned:

### 1. Structured review analysis
- `with_structured_output(Schema)` + `.batch()` turns a pile of reviews into a table in one call per review
- A `Literal` field gives code a value it can safely branch on

### 2. Checking the analysis
- Compare the model's labels with a few hand labels before letting them drive actions

### 3. Replies routed by sentiment
- When the routing signal is already structured, route in code (a dict) — no extra LLM call
- Required parts of an email (greeting, signature, tone, length) are checked in code

### 4. An instruction router
- Classify free-text requests into a closed set of intents, and always include an "other" intent for requests the tool does not handle

### Next Steps
- **`07_multilingual_it_support_desk.ipynb`**: support tickets in ten languages — one structured call per ticket versus a four-step translation chain